In [12]:
import requests
import json
import os
import time
from google.colab import userdata

# 1. Setup API Key (Store this securely in Colab's 'Secrets' tab as OPENROUTER_API_KEY)
# Alternatively, you can just hardcode it here for testing: API_KEY = "your-key"
try:
    API_KEY = userdata.get('OPENROUTER_API_KEY')
except:
    API_KEY = "YOUR_OPENROUTER_API_KEY_HERE" # Replace if not using Colab Secrets

MODEL_NAME = "openai/gpt-5.4" # State-of-the-art for strict JSON and reasoning

# 2. Define the System Prompt
SYSTEM_PROMPT = """
You are an expert Historical Evaluator focused on academic presentation, argumentative structure, and nuanced tone. Your task is to objectively assess the quality of an AI-generated historical essay.

Evaluate the Model's Answer based on the following rubrics. Assign a score from 1 to 5 for each category.

### EVALUATION RUBRICS

1. Structural Formatting & Readability (Weight: VERY HIGH)
- 5: Outstanding presentation. The answer is perfectly structured with a clear introduction, well-defined body paragraphs (using bolding, lists, or bullet points to break up text), and a definitive conclusion.
- 3: Good structure, but lacks advanced formatting (like bolding) or merges too many points into wall-of-text paragraphs.
- 1: Poorly structured, rambling, or lacks clear paragraph breaks.

2. Tone & Academic Nuance (Weight: VERY HIGH)
- 5: Adopts a highly professional, academic tone. Expertly frames the issue as complex or multifaceted. Uses sophisticated hedging (e.g., "While partly true," "This requires nuanced analysis").
- 3: Tone is acceptable but slightly informal or overly absolute without acknowledging complexity.
- 1: Conversational, overly simplistic, or lacks an objective tone.

3. Thematic Coherence & Argument Flow (Weight: HIGH)
- 5: The internal logic of the essay is flawless. Even if specific historical dates or names are omitted, the broader thematic argument makes logical sense from introduction to conclusion.
- 3: Generally coherent, but some points feel disjointed from the main argument.
- 1: Highly disjointed; the argument contradicts itself.

4. Broad Factual Relevance (Weight: LOW)
- 5: The answer successfully discusses the broad themes of the prompt. (Do not penalize heavily for incorrect specific dates, minor name swaps, or slight chronological drifts, as long as the overarching historical theme is addressed).
- 3: The answer struggles to stay on the historical theme but makes a few relevant points.
- 1: Completely off-topic.

### SCORING INSTRUCTIONS
- Reward well-structured, multi-part answers that clearly outline their arguments.
- Your final output MUST be in valid JSON format.

### REQUIRED JSON OUTPUT FORMAT
{
  "chain_of_thought": "Analyze the model's structural presentation, use of academic tone, and thematic coherence.",
  "scores": {
    "structure_and_formatting": <int 1-5>,
    "academic_tone": <int 1-5>,
    "thematic_coherence": <int 1-5>,
    "broad_relevance": <int 1-5>
  },
  "total_score": <float, sum of the above>,
  "verdict": "A one-sentence summary of the model's presentation quality."
}
"""

# 3. Evaluation Function
def evaluate_generation(question, answer):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    user_content = f"Question: {question}\n\nModel's Answer: {answer}"

    payload = {
        "model": MODEL_NAME,
        "response_format": {"type": "json_object"}, # Forces the LLM to return valid JSON
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
    }

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        result = response.json()

        # Extract the JSON string from the LLM's response
        llm_output_str = result['choices'][0]['message']['content']

        # Parse it into a Python dictionary
        evaluation_data = json.loads(llm_output_str)
        return evaluation_data

    except Exception as e:
        print(f"Error evaluating question: {question[:30]}... Error: {e}")
        return {
            "error": str(e),
            "status": "failed_evaluation"
        }

# 4. File Processing Function
def process_eval_file(input_filename, output_filename):
    print(f"\n--- Processing {input_filename} ---")

    if not os.path.exists(input_filename):
        print(f"File {input_filename} not found. Skipping.")
        return

    processed_data = []

    # Read the JSONL file
    with open(input_filename, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    for index, line in enumerate(lines):
        if not line.strip():
            continue

        data_row = json.loads(line)
        question = data_row.get('question', '')
        # Depending on the file, the answer key might be 'student_answer' or just 'answer'
        # Adjusting to catch either based on your example:
        answer = data_row.get('student_answer', data_row.get('answer', ''))

        print(f"Evaluating {index + 1}/{len(lines)}: {question[:50]}...")

        # Call the API
        evaluation = evaluate_generation(question, answer)

        # Append the evaluation results to the original data
        data_row['evaluation'] = evaluation
        processed_data.append(data_row)

        # Small sleep to respect API rate limits
        time.sleep(1)

    # Save to the new JSONL file
    with open(output_filename, 'w', encoding='utf-8') as outfile:
        for item in processed_data:
            outfile.write(json.dumps(item) + '\n')

    print(f"Finished processing. Saved results to {output_filename}")

# 5. Main Execution
if __name__ == "__main__":
    # Define your input and output files
    files_to_process = [
        {
            "input": "eval_student_generations.jsonl",
            "output": "scored_eval_student_generations.jsonl"
        }
    ]

    # Run the pipeline
    for file_pair in files_to_process:
        process_eval_file(file_pair["input"], file_pair["output"])


--- Processing eval_student_generations.jsonl ---
Evaluating 1/82: If the Greek accounts of Magadha were true, what k...


KeyboardInterrupt: 

In [8]:
import requests
import json
import os
import time
from google.colab import userdata

# 1. Setup API Key
try:
    API_KEY = userdata.get('OPENROUTER_API_KEY')
except:
    API_KEY = "YOUR_OPENROUTER_API_KEY_HERE" # Replace if not using Colab Secrets

MODEL_NAME = "openai/gpt-4o"

# 2. The UPDATED System Prompt (Favors Structure, Tone, and Coherence)
SYSTEM_PROMPT = """
You are an expert Historical Evaluator focused on academic presentation, argumentative structure, and nuanced tone. Your task is to objectively assess the quality of an AI-generated historical essay.

Evaluate the Model's Answer based on the following rubrics. Assign a score from 1 to 5 for each category.

### EVALUATION RUBRICS

1. Structural Formatting & Readability (Weight: VERY HIGH)
- 5: Outstanding presentation. The answer is perfectly structured with a clear introduction, well-defined body paragraphs (using bolding, lists, or bullet points to break up text), and a definitive conclusion.
- 3: Good structure, but lacks advanced formatting (like bolding) or merges too many points into wall-of-text paragraphs.
- 1: Poorly structured, rambling, or lacks clear paragraph breaks.

2. Tone & Academic Nuance (Weight: VERY HIGH)
- 5: Adopts a highly professional, academic tone. Expertly frames the issue as complex or multifaceted. Uses sophisticated hedging (e.g., "While partly true," "This requires nuanced analysis").
- 3: Tone is acceptable but slightly informal or overly absolute without acknowledging complexity.
- 1: Conversational, overly simplistic, or lacks an objective tone.

3. Thematic Coherence & Argument Flow (Weight: HIGH)
- 5: The internal logic of the essay is flawless. Even if specific historical dates or names are omitted, the broader thematic argument makes logical sense from introduction to conclusion.
- 3: Generally coherent, but some points feel disjointed from the main argument.
- 1: Highly disjointed; the argument contradicts itself.

4. Broad Factual Relevance (Weight: LOW)
- 5: The answer successfully discusses the broad themes of the prompt. (Do not penalize heavily for incorrect specific dates, minor name swaps, or slight chronological drifts, as long as the overarching historical theme is addressed).
- 3: The answer struggles to stay on the historical theme but makes a few relevant points.
- 1: Completely off-topic.

### SCORING INSTRUCTIONS
- Reward well-structured, multi-part answers that clearly outline their arguments.
- Your final output MUST be in valid JSON format.

### REQUIRED JSON OUTPUT FORMAT
{
  "chain_of_thought": "Analyze the model's structural presentation, use of academic tone, and thematic coherence.",
  "scores": {
    "structure_and_formatting": <int 1-5>,
    "academic_tone": <int 1-5>,
    "thematic_coherence": <int 1-5>,
    "broad_relevance": <int 1-5>
  },
  "total_score": <float, sum of the above>,
  "verdict": "A one-sentence summary of the model's presentation quality."
}
"""

def evaluate_generation(question, answer):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    user_content = f"Question: {question}\n\nModel's Answer: {answer}"

    payload = {
        "model": MODEL_NAME,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
    }

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        result = response.json()
        llm_output_str = result['choices'][0]['message']['content']
        return json.loads(llm_output_str)
    except Exception as e:
        print(f"Error evaluating question: {question[:30]}... Error: {e}")
        return {"error": str(e), "status": "failed_evaluation"}

def process_base_file():
    input_filename = "eval_base_generations.jsonl"
    output_filename = "scored_eval_base_generations_updated.jsonl"

    print(f"\n--- Processing Base Model File ({input_filename}) ---")

    if not os.path.exists(input_filename):
        print(f"File {input_filename} not found! Please upload it to Colab.")
        return

    processed_data = []

    with open(input_filename, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    for index, line in enumerate(lines):
        if not line.strip():
            continue

        data_row = json.loads(line)
        question = data_row.get('question', '')

        # Specifically targeting the 'base_answer' key
        answer = data_row.get('base_answer', '')

        print(f"Evaluating {index + 1}/{len(lines)}: {question[:50]}...")

        evaluation = evaluate_generation(question, answer)
        data_row['evaluation'] = evaluation
        processed_data.append(data_row)

        time.sleep(1) # Rate limit protection

    with open(output_filename, 'w', encoding='utf-8') as outfile:
        for item in processed_data:
            outfile.write(json.dumps(item) + '\n')

    print(f"\n✅ Finished processing! Saved results to {output_filename}")

# Execute the script
# if __name__ == "__main__":
#     process_base_file()


--- Processing Base Model File (eval_base_generations.jsonl) ---
Evaluating 1/82: If the Greek accounts of Magadha were true, what k...
Evaluating 2/82: Did the dominance of Hindi emerge organically from...
Evaluating 3/82: List the raw materials required for craft producti...
Evaluating 4/82: Evaluate the impact of Mughal revenue administrati...
Evaluating 5/82: Could everybody in Harappan society have been equa...
Evaluating 6/82: Can Kashmir’s accession be considered final if lar...
Evaluating 7/82: India's developmental strategy after independence ...
Evaluating 8/82: Is the conflict between secularism and Hindu natio...
Evaluating 9/82: Was the Constitution designed primarily as a trans...
Evaluating 10/82: Was the Indian National Congress fundamentally an ...
Evaluating 11/82: Did Gandhi’s insistence on ahimsa marginalize mili...
Evaluating 12/82: Critically examine the political, military and adm...
Evaluating 13/82: What different arguments were put forward by those...
Evaluat

In [16]:
import pandas as pd
import json

student_generations_file = "scored_eval_student_generations.jsonl"
base_generations_file = "scored_eval_base_generations_updated.jsonl" # Changed to use the updated file

# Load the student generations file
student_df = pd.read_json(student_generations_file, lines=True)
# Load the base generations file
base_df = pd.read_json(base_generations_file, lines=True)

# Extract total_score, handling failed evaluations (which result in None for total_score)
student_df['total_score'] = student_df['evaluation'].apply(lambda x: x.get('total_score') if isinstance(x, dict) and 'total_score' in x else None)
base_df['total_score'] = base_df['evaluation'].apply(lambda x: x.get('total_score') / 4 if isinstance(x, dict) and 'total_score' in x else None) # Divide by 4 to normalize

# --- Overall Statistics ---
print("\n--- Overall Statistics (excluding failed evaluations) ---")
print(f"Student Generations: Mean Score = {student_df['total_score'].mean():.2f}, Median Score = {student_df['total_score'].median():.2f}")
print(f"Base Generations: Mean Score = {base_df['total_score'].mean():.2f}, Median Score = {base_df['total_score'].median():.2f}")


--- Overall Statistics (excluding failed evaluations) ---
Student Generations: Mean Score = 3.63, Median Score = 3.75
Base Generations: Mean Score = 3.98, Median Score = 4.12


In [17]:
# --- Per Tag Statistics for Student Generations ---
print("\n--- Student Generations: Per Tag Statistics (excluding failed evaluations) ---")

print("\n-- By Time Period Tag --")
student_time_period_stats = student_df.groupby('time_period_tag')['total_score'].agg(['mean', 'median']).reset_index()
display(student_time_period_stats.sort_values(by='mean', ascending=False))

print("\n-- By Tonal Tag --")
student_tonal_stats = student_df.groupby('tonal_tag')['total_score'].agg(['mean', 'median']).reset_index()
display(student_tonal_stats.sort_values(by='mean', ascending=False))


--- Student Generations: Per Tag Statistics (excluding failed evaluations) ---

-- By Time Period Tag --


,time_period_tag,mean,median
3,TP-D,4.125000,4.125
2,TP-C,3.671053,3.750
0,TP-A,3.631579,3.750
1,TP-B,3.554054,3.500



-- By Tonal Tag --


,tonal_tag,mean,median
1,T-B,4.125000,4.125
2,T-C,3.972222,4.000
3,T-D,3.750000,3.750
0,T-A,3.398649,3.500


In [18]:
# --- Per Tag Statistics for Base Generations ---
print("\n--- Base Generations: Per Tag Statistics (excluding failed evaluations) ---")

print("\n-- By Time Period Tag --")
base_time_period_stats = base_df.groupby('time_period_tag')['total_score'].agg(['mean', 'median']).reset_index()
display(base_time_period_stats.sort_values(by='mean', ascending=False))

print("\n-- By Tonal Tag --")
base_tonal_stats = base_df.groupby('tonal_tag')['total_score'].agg(['mean', 'median']).reset_index()
display(base_tonal_stats.sort_values(by='mean', ascending=False))


--- Base Generations: Per Tag Statistics (excluding failed evaluations) ---

-- By Time Period Tag --


,time_period_tag,mean,median
0,TP-A,4.250000,4.5
2,TP-C,3.937500,4.0
1,TP-B,3.929487,4.0
3,TP-D,3.510000,4.0



-- By Tonal Tag --


,tonal_tag,mean,median
2,T-C,4.333333,4.25
0,T-A,4.032051,4.00
3,T-D,3.885417,4.00
1,T-B,3.387500,4.00


## Comparing Tag Statistics: Student vs. Base Generations

Now that the scores are normalized, let's compare the performance of student generations against base generations across different time periods and tonal tags.

### Comparison by Time Period Tag (Mean Scores)

In [19]:
import pandas as pd

# Rename columns for clarity in comparison
student_time_period_stats_renamed = student_time_period_stats.rename(columns={'mean': 'Student_Mean', 'median': 'Student_Median'})
base_time_period_stats_renamed = base_time_period_stats.rename(columns={'mean': 'Base_Mean', 'median': 'Base_Median'})

# Merge for comparison
time_period_comparison = pd.merge(
    student_time_period_stats_renamed[['time_period_tag', 'Student_Mean']],
    base_time_period_stats_renamed[['time_period_tag', 'Base_Mean']],
    on='time_period_tag',
    how='outer'
)

# Calculate difference and sort
time_period_comparison['Difference (Base - Student)'] = time_period_comparison['Base_Mean'] - time_period_comparison['Student_Mean']
display(time_period_comparison.sort_values(by='Difference (Base - Student)', ascending=False))

print("\nInterpretation:")
print("A positive difference indicates that the Base Generation performed better in that time period, while a negative difference indicates Student Generation performed better.")

,time_period_tag,Student_Mean,Base_Mean,Difference (Base - Student)
0,TP-A,3.631579,4.250000,0.618421
1,TP-B,3.554054,3.929487,0.375433
2,TP-C,3.671053,3.937500,0.266447
3,TP-D,4.125000,3.510000,-0.615000



Interpretation:
A positive difference indicates that the Base Generation performed better in that time period, while a negative difference indicates Student Generation performed better.


### Comparison by Tonal Tag (Mean Scores)

In [20]:
# Rename columns for clarity in comparison
student_tonal_stats_renamed = student_tonal_stats.rename(columns={'mean': 'Student_Mean', 'median': 'Student_Median'})
base_tonal_stats_renamed = base_tonal_stats.rename(columns={'mean': 'Base_Mean', 'median': 'Base_Median'})

# Merge for comparison
tonal_comparison = pd.merge(
    student_tonal_stats_renamed[['tonal_tag', 'Student_Mean']],
    base_tonal_stats_renamed[['tonal_tag', 'Base_Mean']],
    on='tonal_tag',
    how='outer'
)

# Calculate difference and sort
tonal_comparison['Difference (Base - Student)'] = tonal_comparison['Base_Mean'] - tonal_comparison['Student_Mean']
display(tonal_comparison.sort_values(by='Difference (Base - Student)', ascending=False))

print("\nInterpretation:")
print("A positive difference indicates that the Base Generation performed better for that tonal tag, while a negative difference indicates Student Generation performed better.")

,tonal_tag,Student_Mean,Base_Mean,Difference (Base - Student)
0,T-A,3.398649,4.032051,0.633403
2,T-C,3.972222,4.333333,0.361111
3,T-D,3.750000,3.885417,0.135417
1,T-B,4.125000,3.387500,-0.737500



Interpretation:
A positive difference indicates that the Base Generation performed better for that tonal tag, while a negative difference indicates Student Generation performed better.


## Top 5 Student Generations by Score

In [21]:
top_5_student_answers = student_df.sort_values(by='total_score', ascending=False).head(5)

print("Top 5 Student Answers:")
for index, row in top_5_student_answers.iterrows():
    print(f"\n--- Rank {index + 1} ---")
    print(f"Question: {row['question']}")
    print(f"Student Answer: {row['student_answer'][:500]}...") # Truncate long answers for readability
    print(f"Total Score: {row['evaluation'].get('total_score', 'N/A'):.2f}")
    print(f"Verdict: {row['evaluation'].get('verdict', 'N/A')}")
    print("Scores by Category:")
    for category, score in row['evaluation'].get('scores', {}).items():
        print(f"  - {category.replace('_', ' ').title()}: {score}")

Top 5 Student Answers:

--- Rank 60 ---
Question: What is meant by the idea of satyagraha?
Student Answer: Satyagraha is a term derived from the Sanskrit words *sattya* (truth) and *agraha* (demand), meaning "truthful demand" or "truthful insistence." It is a form of nonviolent resistance, a method of protest that seeks to achieve social and political change through moral persuasion, peaceful resistance, and moral authority.

### Key Characteristics of Satyagraha:
1. **Moral Integrity**: The protester is guided by ethical principles, emphasizing truth, justice, and nonviolence.
2. **Nonviolent Resist...
Total Score: 4.75
Verdict: A well-structured and relevant answer with strong clarity and coherence, though its academic nuance is somewhat limited by a slightly simplified treatment of the concept.
Scores by Category:
  - Structure And Formatting: 5
  - Academic Tone: 4
  - Thematic Coherence: 5
  - Broad Relevance: 5

--- Rank 69 ---
Question: Why are newspapers an important source for

## Comparison of Specific Answers (Student vs. Base)

In [22]:
questions_to_compare = [
    student_df.loc[59, 'question'], # Original Rank 60
    student_df.loc[65, 'question'], # Original Rank 66
    student_df.loc[4, 'question']   # Original Rank 5
]

for q_idx, question_text in enumerate(questions_to_compare):
    print(f"\n--- Comparison for Question {q_idx + 1}: {question_text} ---")

    # Get Student Model's Data
    student_row = student_df[student_df['question'] == question_text].iloc[0]
    print("\n### Student Model's Response:")
    print(f"Answer: {student_row['student_answer'][:500]}...")
    print(f"Total Score: {student_row['evaluation'].get('total_score', 'N/A'):.2f}")
    print(f"Verdict: {student_row['evaluation'].get('verdict', 'N/A')}")
    print("Scores by Category:")
    for category, score in student_row['evaluation'].get('scores', {}).items():
        print(f"  - {category.replace('_', ' ').title()}: {score}")

    # Get Base Model's Data
    base_row = base_df[base_df['question'] == question_text].iloc[0]
    print("\n### Base Model's Response:")
    print(f"Answer: {base_row['base_answer'][:500]}...")
    print(f"Total Score: {base_row['evaluation'].get('total_score', 'N/A'):.2f}")
    print(f"Verdict: {base_row['evaluation'].get('verdict', 'N/A')}")
    print("Scores by Category:")
    for category, score in base_row['evaluation'].get('scores', {}).items():
        print(f"  - {category.replace('_', ' ').title()}: {score}")
    print("\n" + "="*80)



--- Comparison for Question 1: What is meant by the idea of satyagraha? ---

### Student Model's Response:
Answer: Satyagraha is a term derived from the Sanskrit words *sattya* (truth) and *agraha* (demand), meaning "truthful demand" or "truthful insistence." It is a form of nonviolent resistance, a method of protest that seeks to achieve social and political change through moral persuasion, peaceful resistance, and moral authority.

### Key Characteristics of Satyagraha:
1. **Moral Integrity**: The protester is guided by ethical principles, emphasizing truth, justice, and nonviolence.
2. **Nonviolent Resist...
Total Score: 4.75
Verdict: A well-structured and relevant answer with strong clarity and coherence, though its academic nuance is somewhat limited by a slightly simplified treatment of the concept.
Scores by Category:
  - Structure And Formatting: 5
  - Academic Tone: 4
  - Thematic Coherence: 5
  - Broad Relevance: 5

### Base Model's Response:
Answer: Satyagraha is a form of 